# What decides rating once money stops mattering?

In Kaggriculture your score *is* your bank balance — the engine sets the reward to
exactly the money on your farm at the final step. So you would expect the leaderboard
to be a ranking of how much money agents make.

It is, up to about a rating of 1000. After that the relationship falls apart: the
median agent at 3000 banks barely more than the median agent at 1000. Something else
is deciding those two thousand rating points, and this notebook goes looking for it in
the public episode record — 44,000 ladder games with per-agent features attached.

The plan:

1. Show the saturation
2. Ask what still correlates with rating above 1000, across everything the data has
3. Run the same analysis below 1000 as a control, where money should dominate
4. A verdict, including the one control that turned out to be unusable

Nothing here is competition advice. It is a regression on public data plus one line
of the engine.

> **Version pin.** The ladder runs `kaggle-environments` **1.32.4** and older builds
> differ, so we pin it before quoting anything from the referee.

In [ ]:
%pip install -q "kaggle-environments==1.32.4"

In [ ]:
import glob, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.2), "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 11,
})

from kaggle_environments.envs.kaggriculture import kaggriculture as K
SRC_LINES = open(K.__file__, encoding="utf-8").read().split("\n")

def show_block(anchor, after=0, before=0, title=None):
    "Print lines around the first line containing `anchor`. Line numbers move\n"
    "between releases, so we look code up by text rather than by range."
    hits = [i for i, l in enumerate(SRC_LINES) if anchor in l]
    if not hits:
        print(f"!! anchor not found in this version: {anchor!r}")
        return
    i = hits[0]
    lo, hi = max(0, i - before), min(len(SRC_LINES), i + after + 1)
    print(f"--- {title or anchor}  [lines {lo+1}-{hi}] ---")
    for j in range(lo, hi):
        print(f"{j + 1:4d} | {SRC_LINES[j].rstrip()}")

show_block('s.reward = float', before=4, after=1,
           title="the reward is the bank balance, nothing else")

In [ ]:
def find_csv(name):
    for pattern in (f"/kaggle/input/**/{name}", f"../input/**/{name}",
                    f"tmp/dsnew/{name}", name):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    raise SystemExit(f"{name} not found — attach georgymamarin/kaggriculture-episodes")

episodes = pd.read_csv(find_csv("episodes.csv"), low_memory=False)
feats    = pd.read_csv(find_csv("episode_features.csv"), low_memory=False)

# The dataset mixes ladder games with self-play validation runs; its README says to
# drop the latter for anything about strength.
episodes = episodes[episodes["type"] == "EPISODE_TYPE_PUBLIC"]

long = []
for seat in (0, 1):
    other = 1 - seat
    part = episodes[["episode_id", f"sub_{seat}", f"bank_{seat}",
                     f"rating_{seat}", f"bank_{other}", f"rating_{other}"]].copy()
    part.columns = ["episode_id", "sub", "money", "rating",
                    "opp_money", "opp_rating"]
    part["seat"] = seat
    long.append(part)
long = pd.concat(long)
df = long.merge(feats, on=["episode_id", "seat"], how="inner").dropna(
    subset=["rating", "money", "opp_money"])
df["margin"] = df.money - df.opp_money
df["won"] = (df.money > df.opp_money).astype(float)
print(f"{len(df):,} ladder agent-episodes | {df['sub'].nunique():,} submissions")

## 1. The saturation

Bands of 500 rating points, median final money in each.

In [ ]:
EDGES = [0, 750, 1250, 1750, 2250, 2750, 10 ** 9]
LABELS = ["~500", "~1000", "~1500", "~2000", "~2500", "3000+"]
df["band"] = pd.cut(df.rating, bins=EDGES, labels=LABELS, right=False)
bands = df.groupby("band", observed=True)["money"].agg(
    n="size", median="median").astype(int)
bands["gain over previous"] = bands["median"].diff().fillna(0).astype(int)
bands

In [ ]:
fig, ax = plt.subplots()
ax.plot(range(len(bands)), bands["median"], "o-", lw=2, color="#2b6cb0")
ax.set_xticks(range(len(bands))); ax.set_xticklabels(bands.index)
ax.set_xlabel("rating band"); ax.set_ylabel("median final money")
ax.yaxis.set_major_formatter(lambda v, p: f"{v:,.0f}")
ax.set_title("Money buys the first thousand rating points, then stops")
ax.annotate("everything after here is flat", xy=(1.1, bands["median"].iloc[1]),
            xytext=(2.3, bands["median"].iloc[1] * .72), fontsize=10,
            arrowprops=dict(arrowstyle="->", color="grey"))
plt.tight_layout(); plt.show()

The first band to the second roughly triples the money. From there to the top —
two thousand more rating points — adds about a seventh. The reward function has not
changed; money simply stops separating agents.

## 2. Choosing the unit of analysis

A trap to avoid first. The rating attached to a row is the rating *after* that game,
so correlating it with what happened in that game partly measures the scoring rule
rather than the agent: win, and your rating goes up.

So the unit here is the **submission**, not the episode. For each submission we take
its mean rating and its mean behaviour across all its games, then correlate across
submissions. That asks a cleaner question: *do agents that behave this way sit
higher?*

In [ ]:
FIELDS = ["money", "opp_money", "margin", "won", "peak_crew", "total_hires",
          "first_land_day", "elbow_day", "tiles_planted",
          "plants_wheat", "plants_melon", "plants_strawberry"]

def by_submission(sub_df, min_games=8):
    "One row per submission: mean rating, mean features, and money dispersion."
    g = sub_df.groupby("sub")
    out = g[FIELDS].mean()
    out["rating"] = g["rating"].mean()
    out["games"] = g.size()
    out["money_sd"] = g["money"].std()
    out["money_cv"] = out.money_sd / out.money
    return out[out.games >= min_games]

def ranked_corr(agg, target="rating"):
    rows = []
    for c in agg.columns:
        if c in (target, "games"):
            continue
        s = agg[[target, c]].dropna()
        if len(s) < 10:
            continue
        rows.append({"field": c, "r": round(s[target].corr(s[c]), 3),
                     "submissions": len(s)})
    return pd.DataFrame(rows).reindex(
        pd.DataFrame(rows).r.abs().sort_values(ascending=False).index
    ).set_index("field")

high = by_submission(df[df.rating >= 750])
print(f"{len(high)} submissions with 8+ games above the ~1000 band")
ranked_corr(high)

**Crew size comes first, ahead of money.** A submission's average peak crew correlates
+0.59 with where it sits, while its average money manages +0.40.

Two entries need reading carefully rather than taking at face value:

* **`opp_money` at +0.45** is not a finding about opponents. Higher-rated agents are
  matched with higher-rated opponents, who bank more. We return to this below.
* **`money_cv` is positive.** If consistency were being rewarded we would expect a
  negative sign here — more variable earnings, lower rating. It goes the other way,
  so on this evidence consistency is not what the ladder is paying for.

## 3. The control: below 1000, money should win

If the method above is sound, running it on the band where money clearly still matters
should put money on top. If it does not, the method is broken and nothing above counts.

In [ ]:
low = by_submission(df[(df.rating >= 500) & (df.rating < 1000)])
print(f"{len(low)} submissions with 8+ games in the 500-1000 band")
ranked_corr(low).head(6)

It does: **money +0.60, first.** The method separates a band where earnings dominate
from one where they do not, which is what makes the section above worth reading.

## 4. Does crew survive controlling for money?

Correlations one at a time cannot tell whether crew size is just a proxy for earning
more. A regression can.

In [ ]:
def zs(s):
    return (s - s.mean()) / s.std(ddof=0)

def fit(agg, cols, target="rating"):
    d = agg[[target] + cols].dropna()
    X = np.column_stack([np.ones(len(d))] + [zs(d[c]).values for c in cols])
    y = zs(d[target]).values
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    r2 = 1 - resid @ resid / (y @ y)
    return {"model": " + ".join(cols), "R2": round(r2, 3),
            **{c: round(b, 3) for c, b in zip(cols, beta[1:])}}

specs = [["money"], ["peak_crew"], ["money", "peak_crew"],
         ["money", "peak_crew", "plants_strawberry"],
         ["money", "peak_crew", "plants_strawberry", "opp_money"]]
pd.DataFrame([fit(high, s) for s in specs]).set_index("model")

Crew survives; money does not.

* Money alone explains **16%** of the variation in rating between submissions.
* Crew alone explains **35%** — more than twice as much.
* Put together, crew keeps a coefficient of **+0.52** and money falls to +0.27.
* Add opponent wealth and **money collapses to +0.01** while crew barely moves.

That last line is the clearest statement of the puzzle. Once you account for the fact
that stronger agents meet richer opponents, an agent's own money tells you almost
nothing extra about its rating. Crew size still does.

## 5. The control that turned out to be unusable

The obvious next step is to control for the opponent's *rating* rather than their
money. It does not work, and the reason is worth showing because it explains several
of the numbers above.

In [ ]:
high2 = by_submission(df[df.rating >= 750].assign(
    opp_rating=df[df.rating >= 750].opp_rating))
high2["opp_rating"] = df[df.rating >= 750].groupby("sub")["opp_rating"].mean()
pd.DataFrame([fit(high2, ["opp_rating"]),
              fit(high2, ["opp_rating", "money"]),
              fit(high2, ["opp_rating", "peak_crew"])]).set_index("model")

In [ ]:
print(f"corr(own rating, opponent rating) = "
      f"{high2[['rating','opp_rating']].dropna().corr().iloc[0,1]:.4f}")
print(f"mean win rate among these submissions = {high.won.mean():.3f}")
print(f"corr(own rating, win rate)            = "
      f"{high[['rating','won']].corr().iloc[0,1]:+.3f}")

A submission's mean opponent rating explains **99.7%** of its own rating. That is not
a control, it is the same variable wearing a hat: the matchmaker pairs like with like
so tightly that who you played *is* where you sit. Put it in a regression and it
absorbs everything, driving crew and money to zero — which says nothing about either.

The same effect explains why win rate is useless here. Among these submissions the
average win rate is close to a coin flip and its correlation with rating is slightly
**negative**. Strong agents are not winning more games; they are winning the same
share against progressively stronger opposition.

## 6. Verdict

**Above roughly 1000, rating is not bought with money — the single best predictor in
this dataset is how many hands the agent hires.**

The numbers, all across submissions with at least eight games:

| | money | crew |
|---|---|---|
| correlation with rating | +0.40 | **+0.59** |
| variance explained alone | 16% | **35%** |
| coefficient, both in | +0.27 | **+0.52** |
| coefficient, controlling for opponent wealth | **+0.01** | **+0.43** |

Together with strawberry planting the dataset's fields reach **R² ≈ 0.47** — so about
half of what separates these agents is visible here, and crew size is the largest
identifiable part of it.

**What this does not say.** Crew size is not established as a *cause*; it may simply
mark agents whose economy is sound enough to afford one, with the real driver being
whatever makes that possible. The data has no field for it. It also carries no
information about animals at all, and none about simultaneous tiles worked, so two
obvious candidates could not be tested even in principle.

The honest summary is that money stops mattering above 1000, one field takes its
place, and about half the story is still missing from the record.

---

### Reproducing this

Attach `georgymamarin/kaggriculture-episodes` and run top to bottom. Numbers in the
prose were written against the snapshot this ran on; the dataset refreshes daily, so
trust the cells over the text.

### Companion notebooks

* [300 melons return 36% of face value](https://www.kaggle.com/code/xreina8/kaggriculture-300-melons-return-36-of-face-value)
  — how the market punishes volume, from inside the engine.
* [Does a farming game get supply and demand right?](https://www.kaggle.com/code/xreina8/does-a-farming-game-get-supply-and-demand-right)
  — the same pricing rule against fifteen years of a real commodity market.